In [ ]:
!pip -q install torchxrayvision captum
!git clone https://github.com/mlmed/gifsplanation

In [ ]:
import sys,os
sys.path.insert(0,"./gifsplanation/")
sys.path.insert(0,"../torchxrayvision/")
import skimage
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import torch, torchvision
import torchxrayvision as xrv
import attribution
import requests
import json
import warnings
warnings.filterwarnings('ignore')

## Descarga de token de kaggle que permite acceder al set de datos
json_response= requests.get("https://raw.github.com/learnradiomics/Image_processing/main/kaggle.json")
token = json.loads(json_response.text)
with open("kaggle.json", "w") as outfile:
    json.dump(token, outfile)

## Carga de datos desde Kaggle
! pip install kaggle
! mkdir ~/.kaggle
! cp kaggle.json ~/.kaggle/
! chmod 600 ~/.kaggle/kaggle.json

## Dataset
! kaggle datasets download hshenriquez/cxr-images-sample
!unzip /content/cxr-images-sample.zip

In [ ]:
device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
device

In [ ]:
def predict_torchxrayvision(img, model):
    import torch

    device = next(model.parameters()).device

    img = img.float().to(device)
    model.eval()

    with torch.no_grad():
        outputs = model(img)

    return outputs.detach().cpu()

def move_gifsplanation_inputs_to_device(image, model, ae, use_cuda=True):
    import torch
    import numpy as np

    device = torch.device("cuda" if use_cuda and torch.cuda.is_available() else "cpu")

    if isinstance(image, np.ndarray):
        image = torch.from_numpy(image).float()

    image = image.float().to(device)
    model = model.to(device)
    ae = ae.to(device)

    model.eval()
    ae.eval()

    print("Using device:", device)
    print("Image:", image.shape, image.device, image.dtype)
    print("Model:", next(model.parameters()).device, next(model.parameters()).dtype)
    print("AE:", next(ae.parameters()).device, next(ae.parameters()).dtype)

    return image, model, ae, device

In [ ]:
### Carga de datos
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cxr_data = np.load('/content/CXR_samples_from_RSNA_Challenge.npy')

print(cxr_data.shape, cxr_data.dtype, cxr_data.min(), cxr_data.max())

In [ ]:
## Vsualización de casos aleatorios

fig, axes1 = plt.subplots(2,3,figsize=(10,10))
for j in range(2):
    for k in range(3):
        i = np.random.randint(0, cxr_data.shape[0])
        axes1[j][k].set_axis_off()
        axes1[j][k].imshow(cxr_data[i,:,:], cmap='gray')

plt.show()

In [ ]:
##Carga modelo y autoencoder
ae = xrv.autoencoders.ResNetAE(weights="101-elastic").to(device)
model = xrv.models.DenseNet(weights="all").to(device)
#model.pathologies

In [ ]:
def prepare_single_cxr_for_xrv(cxr_data, idx, device):
    """
    Toma cxr_data con forma [N, H, W] uint8
    y devuelve una imagen compatible con TorchXRayVision:
    [1, 1, 224, 224]
    """

    img_np = cxr_data[idx].astype(np.float32)

    # Normalización esperada por TorchXRayVision
    img_np = xrv.datasets.normalize(img_np, 255)

    # [H, W] -> [1, 1, H, W]
    img_tensor = torch.from_numpy(img_np).float()
    img_tensor = img_tensor.unsqueeze(0).unsqueeze(0)

    img_tensor = img_tensor.to(device)

    return img_tensor

In [ ]:
idx = 7
img = prepare_single_cxr_for_xrv(cxr_data, idx, device)

print(img.shape)
print(img.dtype)
print(img.device)
print(img.min(), img.max())

plt.figure(figsize=(6,6))
plt.imshow(cxr_data[idx,:,:], cmap='gray')
plt.title("Imagen a evaluar idx:{}".format(idx))
plt.show()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)
img = img.float().to(device)

## Inferencia sobre imagen:
outputs = predict_torchxrayvision(img, model)

scores = outputs[0].numpy()

df_results = (
    pd.DataFrame({
        "Pathology": model.pathologies,
        "Score": scores
    })
    .sort_values("Score", ascending=False)
)

df_results

In [ ]:
target = "Pleural_Thickening"

image, model, ae, device = move_gifsplanation_inputs_to_device(img, model, ae)

%matplotlib inline
attribution.generate_video(
    img,
    model,
    target,
    ae,
    target_filename="test",
    border=False,
    show=True,
    ffmpeg_path="ffmpeg"
)